# Redis Queue Security Lab Report

Muc tieu: demo he thong queue gui email su dung Redis command thu cong (`redis-cli`) va ghi nhan ket qua kiem thu bao mat.

## Kien truc

- `web`: nhan email admin tu form va day job vao Redis bang `LPUSH`.
- `scheduler`: dinh ky tao job daily report va `LPUSH`.
- `worker`: dung `BRPOP` de lay job va gui email qua MailHog.
- `mailhog`: xem email tai `http://localhost:8025`.

In [ ]:
import subprocess

def run_cmd(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
    print(result.stdout.strip() or result.stderr.strip())

print('Containers:')
run_cmd('docker compose ps')

In [ ]:
print('Run defensive security tests:')
run_cmd('python scripts/security_test.py')

In [ ]:
print('Worker logs (tail):')
run_cmd('docker compose logs --tail=30 worker')

## Nhan xet

- Input email sai dinh dang bi chan o web layer.
- Job bi sua/tampered bi worker loai bo boi co che HMAC.
- Job hop le duoc worker xu ly va gui mail thanh cong.

Ket luan: van co the trinh bay kich ban tan cong gia lap, nhung he thong da duoc harden de tranh khai thac truc tiep.

## Ket qua thuc te da ghi nhan

- `docker compose ps`: tat ca services (`redis`, `web`, `worker`, `scheduler`, `mailhog`) dang `Up`.
- `python scripts/security_test.py`:
  - `[test] invalid email blocked: True`
  - `[test] pushed valid signed job`
  - `[test] pushed tampered job`
  - queue depth truoc/sau deu ve `0` (worker xu ly ngay).
- `docker compose logs --tail=60 worker`:
  - `[worker] Sent report to safe-admin@example.com`
  - `[worker] Dropped job with invalid signature`
- MailHog API (`/api/v2/messages`) cho thay email den `safe-admin@example.com` va `admin@example.com`.